# Week 8 - Master notebook: re-run the Week 8 work items and combine the validation
Runs the four Week 8 notebooks in `standalone_question_reports/` (W8-1 supplier reliability, W8-2 supplier risk, W8-3 stockout and reorder, W8-4 customer segments), checks that each reports all checks passing, and writes one combined `validation_report_week08.csv`. All four are required. Each notebook finds the merged data itself (a `merged` folder in this repository, or `CADETX_MERGED`); W8-4 no longer needs the separate clean folder because customer region, type and industry come from the merged sales file. The full run takes about five minutes. If Jupyter's `nbconvert` is not installed the notebooks are run in-process instead.

In [1]:
import subprocess, sys, json, os, io, contextlib, pandas as pd
from pathlib import Path
HERE=Path.cwd(); R=HERE/"standalone_question_reports"
ITEMS=[("W8-1 Supplier reliability","w81_supplier_reliability","w81_supplier_reliability.ipynb","validation_report_week08_w81.csv",29),
       ("W8-2 Supplier risk","w82_supplier_risk","w82_supplier_risk.ipynb","validation_report_week08_w82.csv",38),
       ("W8-3 Stockout and reorder","w83_stockout_reorder","w83_stockout_reorder.ipynb","validation_report_week08_w83.csv",31),
       ("W8-4 Customer segments","w84_customer_segments","w84_customer_segments.ipynb","validation_report_week08_w84.csv",22)]
def run_inprocess(d,nbname):
    nb=json.load(open(d/nbname)); ns={"__name__":"__main__"}; old=os.getcwd(); os.chdir(d); sys.path.insert(0,str(d))
    try:
        for c in nb["cells"]:
            if c["cell_type"]=="code":
                with contextlib.redirect_stdout(io.StringIO()): exec(compile("".join(c["source"]),nbname,"exec"),ns)
    finally:
        os.chdir(old); sys.path.remove(str(d))
parts=[]
for label,folder,nbname,vfile,n_expected in ITEMS:
    d=R/folder
    assert (d/nbname).exists(),f"missing notebook {d/nbname}"
    r=subprocess.run([sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute","--inplace",nbname],cwd=d,capture_output=True,text=True)
    if r.returncode!=0:
        assert "No module named" in r.stderr or "jupyter" in r.stderr.lower() and "not" in r.stderr.lower(),f"{label} failed to run:\n{r.stderr[-1500:]}"
        run_inprocess(d,nbname)
    v=pd.read_csv(d/vfile); assert len(v)==n_expected,f"{label}: expected {n_expected} checks, found {len(v)}"
    v.insert(0,"Work_item",label); parts.append(v); print(f"{label}: {int((v.Status=='PASS').sum())} of {len(v)} checks pass")
V=pd.concat(parts,ignore_index=True); V["Check_ID"]=range(1,len(V)+1)
V=V[["Check_ID","Work_item","Check_Item","Expected","Actual","Status"]]; V.to_csv(HERE/"validation_report_week08.csv",index=False)
assert (V.Status=="PASS").all(),"a check failed"
print(len(V),"checks in all,",int((V.Status=="PASS").sum()),"pass")

W8-1 Supplier reliability: 29 of 29 checks pass
W8-2 Supplier risk: 38 of 38 checks pass
W8-3 Stockout and reorder: 31 of 31 checks pass
W8-4 Customer segments: 22 of 22 checks pass
120 checks in all, 120 pass
